# sum-and-broadcast-duality — faded example 3: mean_back: divide sum_back by N

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-and-broadcast-duality`. Running the beacon reports progress on the `Backprop: sum/broadcast duality` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum/broadcast duality` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-and-broadcast-duality`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-and-broadcast-duality"
DD_SUBTOPIC = "Backprop: sum/broadcast duality"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Because mean is sum scaled by `1/N`, its backward is `sum_back` divided by the reduction count `N = x.shape[dim]`. The constant factor passes straight through the chain rule.

## Faded exercise 3

`sum_back` is provided. Complete `mean_back` by reusing `sum_back` and dividing by the reduction count along `dim`.

**Fill in:** divide sum_back's result by the reduction count x.shape[dim]

In [ ]:
Tensor = t.Tensor


def sum_back(grad_out, out, x, dim, keepdim=False):
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    return grad_out.expand_as(x).clone()


def mean_back(grad_out, out, x, dim, keepdim=False):
    result = None  # TODO: divide sum_back's result by the reduction count x.shape[dim]
    return result


t.manual_seed(0)
x = t.randn(2, 5)
grad_out = t.tensor([1.0, 2.0])
grad_x = mean_back(grad_out, x.mean(dim=1), x, dim=1)

def _test():
    t.manual_seed(0)
    x = t.randn(2, 5)
    grad_out = t.tensor([1.0, 2.0])
    grad_x = mean_back(grad_out, x.mean(dim=1), x, dim=1)
    assert tuple(grad_x.shape) == (2, 5), 'must match x.shape'
    # each entry of row r is grad_out[r] / 5
    assert abs(grad_x[0, 0].item() - 1.0 / 5) < 1e-6
    assert abs(grad_x[1, 0].item() - 2.0 / 5) < 1e-6
    xg = x.clone().requires_grad_(True)
    xg.mean(dim=1).backward(grad_out)
    assert t.allclose(grad_x, xg.grad), 'must match autograd'

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
Tensor = t.Tensor


def sum_back(grad_out, out, x, dim, keepdim=False):
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    return grad_out.expand_as(x).clone()


def mean_back(grad_out, out, x, dim, keepdim=False):
    result = sum_back(grad_out, out, x, dim, keepdim) / x.shape[dim]
    return result


t.manual_seed(0)
x = t.randn(2, 5)
grad_out = t.tensor([1.0, 2.0])
grad_x = mean_back(grad_out, x.mean(dim=1), x, dim=1)
```
</details>